# Decile Analysis for Customer Segmentation and Sales Contribution

In [ ]:
# Purchase history logs for each customer.（Dummy data）
# Including data showing multiple purchases by the same customer, 
# and data containing invalid negative amounts.
sales_logs = [
    {"customer_id": "C001", "amount": 5000},
    {"customer_id": "C002", "amount": 12000},
    {"customer_id": "C003", "amount": 3000},
    {"customer_id": "C001", "amount": 4000}, # Second purchase of C001
    {"customer_id": "C004", "amount": 15000},
    {"customer_id": "C005", "amount": -2000}, # invalid data
    {"customer_id": "C006", "amount": 8000},
    {"customer_id": "C007", "amount": 25000},
    {"customer_id": "C008", "amount": 1000},
    {"customer_id": "C009", "amount": 6000},
    {"customer_id": "C010", "amount": 9000},
    {"customer_id": "C011", "amount": 0},    # invalid data (zero amount excluded)
    {"customer_id": "C012", "amount": 11000},
]

def get_users(logs):
    """
    Exclude amounts that are 0 or less, and calculate the total purchase amount for each customer.

    Args:
        logs (list): Sales logs. Each log is a dictionary containing customer_id and amount.
    
    Returns:
        list: A list of dictionaries containing customer_id and total_amount.
    """
    totals = {}
    for log in logs:
        if log["amount"] > 0:
            customer_id = log["customer_id"]
            if customer_id in totals:
                totals[customer_id] += log["amount"]
            else:
                totals[customer_id] = log["amount"]

    users = [
        {"customer_id": cid, "total_amount": amount}
        for cid, amount in totals.items()
    ]
            
    return users

def decile_division(logs):
    """
    Sort customers in descending order of total purchase amount (highest to lowest),
    and divide the sorted customer list into 10 eaual parts (Decile 1 to Decile 10).

    Args:
        logs(list): Sales logs. Each log is a dictionary containing customer_id and total_amount.
    
    Returns:
        list: A list of lists, where each sublist represents a decile group
        containing dictionaries with customer_id and total_amount.
    """
    users = get_users(logs)
    sorted_users = sorted(users, key=lambda x: x["total_amount"], reverse=True)

    n = len(sorted_users)
    deciles = []
    for i in range(10):
        start = i * n // 10
        end = (i + 1) * n // 10
        decile_group = sorted_users[start:end]
        deciles.append(decile_group)

    return deciles

def aggregate_deciles(logs):
    """
    Calculate the total purchase amount for that group for each decile group.

    Args:
        logs(list): Sales logs Each log is a dictionary containing customer_id and total_amount.

    Returns:
        list: A list of dictionaries containing decile, customer_count, and total_amount.
    """
    deciles = decile_division(logs)
    aggregate = []
    for i, decile in enumerate(deciles):
        total_amount = sum(user["total_amount"] for user in decile)

        aggregate.append({
            "decile": i + 1,
            "customer_count": len(decile),
            "total_amount": total_amount
        })
    
    return aggregate

def calculate_ratio(logs):
    """
    Calculate the sales composition ratio of each decile relative to the total sales,
    and the cumulative sales ratio from the group.
    
    Args:
        logs(list): Sales logs Each log is a dictionary containing customer_id and total_amount.
        
    Returns:
        list: A list of dictionaries containing decile, customer_count, composition_ratio, and cumulative_ratio.
    """
    aggregate = aggregate_deciles(logs)
    total_sales = sum(d["total_amount"] for d in aggregate)

    current_ratio = 0
    for row in aggregate:
        composition_ratio = row["total_amount"] / total_sales
        row["composition_ratio"] = composition_ratio
        cumulative_ratio = composition_ratio + current_ratio
        row["cumulative_ratio"] = cumulative_ratio
        current_ratio += composition_ratio
    
    return aggregate


print(calculate_ratio(sales_logs))





[{'decile': 1, 'customer_count': 1, 'total_amount': 25000, 'composition_ratio': 0.25252525252525254, 'cumulative_ratio': 0.25252525252525254}, {'decile': 2, 'customer_count': 1, 'total_amount': 15000, 'composition_ratio': 0.15151515151515152, 'cumulative_ratio': 0.4040404040404041}, {'decile': 3, 'customer_count': 1, 'total_amount': 12000, 'composition_ratio': 0.12121212121212122, 'cumulative_ratio': 0.5252525252525253}, {'decile': 4, 'customer_count': 1, 'total_amount': 11000, 'composition_ratio': 0.1111111111111111, 'cumulative_ratio': 0.6363636363636365}, {'decile': 5, 'customer_count': 1, 'total_amount': 9000, 'composition_ratio': 0.09090909090909091, 'cumulative_ratio': 0.7272727272727274}, {'decile': 6, 'customer_count': 1, 'total_amount': 9000, 'composition_ratio': 0.09090909090909091, 'cumulative_ratio': 0.8181818181818183}, {'decile': 7, 'customer_count': 1, 'total_amount': 8000, 'composition_ratio': 0.08080808080808081, 'cumulative_ratio': 0.8989898989898991}, {'decile': 8, '